# 04 — Layer-wise activation analysis

Compares teacher (Qwen3-14B) vs a trained student checkpoint's activations on the same probe inputs, per layer, split by attention output vs FFN output (not the combined residual stream), to localize where the representational gap originates — the paper's key mechanistic figure.

**Two probe input sets** (configs/experiment.yaml: `layer_analysis`), compared side by side:
- `tool_use` — real Glaive TEST-split text (held out from training)
- `general` — the same wikitext sample `evaluation/perplexity.py` uses

so divergence that shows up on `general` too, not just `tool_use`, reads as generic distillation drift rather than something tool-use-specific.

**Why both CKA and cosine distance:** teacher (40 layers, hidden_size 5120) and student (36 layers, hidden_size 2560) differ in *both* depth and width — plain cosine similarity between raw activation vectors isn't even defined across mismatched widths. Both metrics here instead operate on (n x n) sample-pairwise structure, which stays valid regardless of width — CKA via Gram matrices, cosine distance via each model's own pairwise-cosine-similarity matrix (RSA-style). See `analysis/layer_analysis.py`'s module docstring for the full reasoning; this is a flagged design decision, not an assumption.

**Memory:** a free-tier T4 (16GB) is tight for holding Qwen3-14B (~7-8GB in 4-bit) and Qwen3-4B (~2-2.5GB in 4-bit) resident *at the same time*, on top of activation memory. This notebook never does — it loads one model, extracts + caches activations for both probe sets to disk, frees it, then does the same for the other model. The comparison step needs only the two cache files, no models loaded at all.

Requires, before running:
- `notebooks/02_training.ipynb` to have produced the student checkpoint you want to compare (`STUDENT_CONDITION` below — `"distilled"` is the primary comparison the research question is about, but any of the three conditions can be loaded here)
- `python -m adbench.data.prepare` (Glaive test split) and `python -m adbench.data.general_eval` (wikitext sample)

All the real logic lives in `adbench.analysis.layer_analysis` and is unit-tested locally with synthetic tensors and a fake model (`tests/test_layer_analysis.py`) — no GPU needed for the metrics themselves; only the extraction steps below need one.

In [ ]:
# Private repo: create a GitHub personal access token (repo scope) and add
# it as a secret named GH_TOKEN before running this cell:
#   - Colab: key icon (Secrets) in the left sidebar, then enable notebook access
#   - Kaggle: Add-ons -> Secrets, then enable it for this notebook
# Skips re-cloning if this session's runtime already has the repo (e.g. you
# ran 00_setup_colab.ipynb earlier in this same session).
import os

# Reduces CUDA OOM from a single large allocation (e.g. loading the 14B
# teacher) by letting the allocator grow a segment incrementally instead of
# needing one big contiguous block upfront.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GH_TOKEN')
    else:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = token
    !git clone https://$GH_TOKEN@github.com/Nahla-Nabil/agentic-distillation-benchmark.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt
!pip install -q -e .

In [ ]:
# Teacher and student share the identical tokenizer (verified —
# scripts/verify_tokenizer_compatibility.py), so one tokenizer, loaded once,
# is reused for both extraction passes below (load_teacher() only returns
# the model, since training.py never needed the teacher's own tokenizer
# object — this is the one place layer analysis needs it explicitly).
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(models_config["student"]["hf_id"])

## Step 1 — teacher: extract & cache, then free

Loads Qwen3-14B, runs both probe sets through it, caches the activations to disk, then deletes the model and clears the CUDA cache — the student is never loaded while the teacher still is.

In [ ]:
import gc

import torch

from adbench.training.train import load_teacher

teacher_model = load_teacher(models_config)
n_teacher_layers = count_layers(teacher_model)
print(f"teacher: {n_teacher_layers} layers")

extract_and_cache_activations(
    teacher_model, tokenizer, tool_use_texts, cache_dir / "teacher_tool_use",
    max_length=la_config["probe_max_length"],
)
extract_and_cache_activations(
    teacher_model, tokenizer, general_texts, cache_dir / "teacher_general",
    max_length=la_config["probe_max_length"],
)

del teacher_model
gc.collect()
torch.cuda.empty_cache()

## Plot

Divergence vs layer depth (student layer index on the x-axis), one line per input set (`tool_use` vs `general`) — a separate panel per stream (attention/FFN) and per metric. This is the key figure: where the `tool_use` line diverges from `general` is where the gap looks tool-use-specific, not just generic distillation drift; where both lines move together, it's likely the latter.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(all_rows)
streams = ["attention", "ffn"]
metrics = la_config["divergence_metrics"]

fig, axes = plt.subplots(len(metrics), len(streams), figsize=(11, 4 * len(metrics)), squeeze=False)
for row_idx, metric in enumerate(metrics):
    for col_idx, stream in enumerate(streams):
        ax = axes[row_idx][col_idx]
        for input_set, color in [("tool_use", "tab:red"), ("general", "tab:blue")]:
            sub = df[(df["stream"] == stream) & (df["input_set"] == input_set)].sort_values("source_layer")
            ax.plot(sub["source_layer"], sub[metric], marker="o", label=input_set, color=color)
        ax.set_title(f"{stream} — {metric}")
        ax.set_xlabel("student layer index")
        ax.set_ylabel(metric)
        ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from adbench.evaluation.run_eval import load_condition_model

student_model, _student_tokenizer = load_condition_model(STUDENT_CONDITION, experiment_config, models_config)
n_student_layers = count_layers(student_model)
print(f"{STUDENT_CONDITION}: {n_student_layers} layers")

extract_and_cache_activations(
    student_model, tokenizer, tool_use_texts, cache_dir / "student_tool_use",
    max_length=la_config["probe_max_length"],
)
extract_and_cache_activations(
    student_model, tokenizer, general_texts, cache_dir / "student_general",
    max_length=la_config["probe_max_length"],
)

del student_model
gc.collect()
torch.cuda.empty_cache()

## Step 3 — compare (no models loaded — just the two cache files)

In [ ]:
# student layer i -> teacher layer align_layers[i] (configs/models.yaml: layer_alignment)
layer_alignment = align_layers(n_student_layers, n_teacher_layers)

rows_tool_use = compare_cached_activations(
    cache_dir / "student_tool_use", cache_dir / "teacher_tool_use",
    layer_alignment, metrics=tuple(la_config["divergence_metrics"]), input_set="tool_use",
)
rows_general = compare_cached_activations(
    cache_dir / "student_general", cache_dir / "teacher_general",
    layer_alignment, metrics=tuple(la_config["divergence_metrics"]), input_set="general",
)
all_rows = rows_tool_use + rows_general

write_layer_analysis_results(all_rows, REPO_ROOT / la_config["results_path"])
print(f"{len(all_rows)} rows written to {la_config['results_path']}")

## Plot

TODO: divergence (CKA / cosine distance) vs layer depth, one line for attention outputs and one for FFN outputs — the key figure for the "which layers" half of the research question.